# E3 ablation preparation

CPU-only preparation for comparing full E2 with E2 without abstention. This notebook does not overwrite the E2 reference outputs. The later generation stage requires the GPU cluster.

In [ ]:
from pathlib import Path
import hashlib, json
from collections import Counter, defaultdict

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "E2_Qwen_F1").exists():
    REPO_ROOT = REPO_ROOT.parent
E2_RESULTS = REPO_ROOT / "E2_Qwen_F1" / "results_e2_qwen_f1"
E3_RESULTS = REPO_ROOT / "results_e3_ablation"
E3_RESULTS.mkdir(exist_ok=True)
def read_jsonl(path):
    if not path.exists():
        raise FileNotFoundError(path)
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

def read_json(path):
    if not path.exists():
        raise FileNotFoundError(path)
    return json.loads(path.read_text(encoding="utf-8"))
paths = {name: E2_RESULTS / name for name in (
    "shared_contexts.jsonl", "shared_contexts_manifest.json",
    "rgb_e2_passage_scores.jsonl", "rgb_e2_e0_predictions.jsonl",
    "rgb_e2_e2_predictions.jsonl", "rgb_e2_e0_predictions.jsonl.meta.json",
    "rgb_e2_e2_predictions.jsonl.meta.json")}
contexts = read_jsonl(paths["shared_contexts.jsonl"])
manifest = read_json(paths["shared_contexts_manifest.json"])
scores = read_jsonl(paths["rgb_e2_passage_scores.jsonl"])
e0 = read_jsonl(paths["rgb_e2_e0_predictions.jsonl"])
e2 = read_jsonl(paths["rgb_e2_e2_predictions.jsonl"])
e0_meta = read_json(paths["rgb_e2_e0_predictions.jsonl.meta.json"])
e2_meta = read_json(paths["rgb_e2_e2_predictions.jsonl.meta.json"])
print({"contexts": len(contexts), "scores": len(scores), "e0": len(e0), "e2_cache": len(e2)})

In [ ]:
def row_key(row):
    return (row["condition"], str(row["id"]))

def context_hash(rows):
    text = "".join(json.dumps(row, ensure_ascii=False, sort_keys=True, separators=(",", ":")) + "\n" for row in rows)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

expected = {"noise_0.0": 300, "noise_0.4": 300, "noise_0.8": 300, "incomplete": 300, "contradictory": 100}
test_expected = {"noise_0.0": 210, "noise_0.4": 210, "noise_0.8": 210, "incomplete": 210, "contradictory": 70}
assert Counter(row["condition"] for row in contexts) == Counter(expected)
assert context_hash(contexts) == manifest["sha256_canonical_jsonl"]
assert manifest["seed"] == 42 and manifest["calibration_fraction"] == 0.3
doc_count_distribution = Counter(len(row["docs"]) for row in contexts)
print("Passage-count distribution:", dict(doc_count_distribution))
assert all(sum(row["condition"] == c and row["split"] == "test" for row in contexts) == n for c, n in test_expected.items())
context_keys = {row_key(row) for row in contexts}
assert len(context_keys) == len(contexts)

def check_predictions(rows, name, complete):
    keys = [row_key(row) for row in rows]
    assert not [k for k, n in Counter(keys).items() if n > 1], f"Duplicate {name} rows"
    missing = context_keys - set(keys)
    assert not (set(keys) - context_keys), f"Unexpected {name} rows"
    if complete:
        assert not missing, f"Incomplete {name}: {len(missing)} missing"
    print(f"{name}: {len(rows)} rows; missing={len(missing)}")
    return missing

check_predictions(e0, "E0", complete=True)
e2_missing = check_predictions(e2, "cached E2", complete=False)
assert e0_meta["context_sha256"] == e2_meta["context_sha256"] == manifest["sha256_canonical_jsonl"]
assert e0_meta["seed"] == e2_meta["seed"] == manifest["seed"]
print("Shared context, split, and metadata checks: OK")
if e2_missing:
    print("WARNING: cached E2 predictions are partial and cannot be the final E2 reference.")

In [ ]:
score_groups = defaultdict(list)
for row in scores:
    context = next(item for item in contexts if row_key(item) == row_key(row))
    assert 0 <= row["doc_idx"] < len(context["docs"])
    score_groups[row_key(row)].append(row)
context_doc_counts = {row_key(row): len(row["docs"]) for row in contexts}
incomplete_score_groups = [(k, len(v), context_doc_counts[k]) for k, v in score_groups.items() if len(v) != context_doc_counts[k]]
missing_score_groups = [(k, 0) for k in context_keys if k not in score_groups]
score_gaps = incomplete_score_groups + missing_score_groups
if score_gaps:
    print("ERROR: passage-score cache is incomplete:", len(score_gaps), "context groups")
    print("First gaps:", score_gaps[:5])
    raise RuntimeError("Regenerate the missing Qwen passage scores before the E3 generation run.")
print("Passage-score alignment: OK; scores match every context's passage count")

filter_threshold = e2_meta["filter_threshold"]
abstain_threshold = e2_meta["abstain_threshold"]
contexts_by_key = {row_key(row): row for row in contexts}
prepared = {}
for k, context in contexts_by_key.items():
    rows = sorted(score_groups[k], key=lambda x: x["doc_idx"])
    kept = [row["doc_idx"] for row in rows if row["score"] >= filter_threshold]
    if not kept:
        kept = [max(rows, key=lambda x: x["score"])["doc_idx"]]
    prepared[k] = {
        "condition": context["condition"], "id": context["id"], "query": context["query"],
        "split": context["split"], "docs": context["docs"], "kept_indices": kept,
        "kept_docs": [context["docs"][i] for i in kept],
        "scores": [row["score"] for row in rows],
        "relevant": [row["relevant"] for row in rows],
        "max_original_score": max(row["score"] for row in rows),
        "would_abstain": max(row["score"] for row in rows) < abstain_threshold,
    }
print("Prepared contexts:", len(prepared))
print("Filter threshold:", filter_threshold, " Abstention threshold:", abstain_threshold)

In [ ]:
prepared_path = E3_RESULTS / "prepared_filtered_contexts.jsonl"
with prepared_path.open("w", encoding="utf-8") as handle:
    for k in sorted(prepared):
        handle.write(json.dumps(prepared[k], ensure_ascii=False) + "\n")
config = {
    "context_sha256": manifest["sha256_canonical_jsonl"], "seed": manifest["seed"],
    "model": e2_meta["model"], "max_new_tokens": e2_meta["max_new_tokens"],
    "filter_threshold": filter_threshold, "abstain_threshold": abstain_threshold,
    "method": "E2_without_abstention",
    "generation_rule": "generate from kept_docs for every row; never replace generation with refusal",
    "cached_e2_rows": len(e2), "cached_e2_complete": not bool(e2_missing),
}
(E3_RESULTS / "e3_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
print("Wrote", prepared_path)
print("Wrote", E3_RESULTS / "e3_config.json")